In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os
import time
from imblearn.over_sampling import SMOTE

In [2]:
# Constants
RANDOM_STATE = 1082
FILE_PATH: str = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "StackedEnsemble_RF (SMOTE)"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [3]:
# Preprocessing
df = pd.read_csv(FILE_PATH)
# Prepare features and labels
X = df.drop(columns=["label", "timestamp"])
y = df["label"].map({"benign": 0, "malicious": 1})  # encode labels

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=RANDOM_STATE, stratify=y
)

# Unsupervised stages see only the benign rows of the training split
X_fit_raw = X_train_raw[y_train == 0]

scaler = StandardScaler()
scaler.fit(X_fit_raw)
X_fit_scaled = scaler.transform(X_fit_raw)
X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
pca_svm_lof.fit(X_fit_scaled)
X_fit_pca_svm_lof = pca_svm_lof.transform(X_fit_scaled)
X_train_pca_svm_lof = pca_svm_lof.transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
pca_ae.fit(X_fit_scaled)
X_fit_pca_ae = pca_ae.transform(X_fit_scaled)
X_train_pca_ae = pca_ae.transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

In [4]:
# Train Base Models
start_time_total = time.time()

# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_fit_pca_svm_lof)

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_fit_pca_svm_lof)

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
ae.fit(X_fit_pca_ae, X_fit_pca_ae)

# Threshold based on 97.2% percentile from training reconstruction error
fit_recon = ae.predict(X_fit_pca_ae)
fit_error = np.mean((X_fit_pca_ae - fit_recon) ** 2, axis=1)
threshold = np.percentile(fit_error, 97.2)


def stack_predictions(X_pca_svm_lof, X_pca_ae):
    recon = ae.predict(X_pca_ae)
    recon_error = np.mean((X_pca_ae - recon) ** 2, axis=1)
    return np.vstack([
        np.where(ocsvm.predict(X_pca_svm_lof) == -1, 1, 0),
        np.where(lof.predict(X_pca_svm_lof) == -1, 1, 0),
        np.where(recon_error > threshold, 1, 0),
    ]).T


X_meta_train = stack_predictions(X_train_pca_svm_lof, X_train_pca_ae)
base_train_time = time.time() - start_time_total

X_meta_test = stack_predictions(X_test_pca_svm_lof, X_test_pca_ae)
y_meta_train = y_train
y_meta_test = y_test

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [5]:
# Meta-Classifier Ensemble
start_meta = time.time()

# Set up SMOTE sampling strategy for all minority classes
unique, counts = np.unique(y_meta_train, return_counts=True)
majority_class = unique[np.argmax(counts)]
count_of_majority_class = counts[np.argmax(counts)]
desired_strategy = {cls: count_of_majority_class for cls in unique if cls != majority_class}

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=desired_strategy)
X_meta_train_bal, y_meta_train_bal = smote.fit_resample(X_meta_train, y_meta_train)

# Meta-classifier (Random Forest)
meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
meta_clf.fit(X_meta_train_bal, y_meta_train_bal)
train_time = base_train_time + (time.time() - start_meta)

stacked_pred = meta_clf.predict(X_meta_test)


def predict_end_to_end(rows):
    rows_scaled = scaler.transform(rows)
    return meta_clf.predict(stack_predictions(pca_svm_lof.transform(rows_scaled),
                                              pca_ae.transform(rows_scaled)))

In [6]:
# Evaluation
report = classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1],  # 0 = benign, 1 = malicious
    target_names=["benign", "malicious"],
    zero_division=0,
    output_dict=True
)
f1 = report["weighted avg"]["f1-score"]
accuracy = accuracy_score(y_meta_test, stacked_pred)
print(classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1],
    target_names=["benign", "malicious"],
    zero_division=0
))
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

      benign       1.00      0.99      1.00       625
   malicious       0.95      1.00      0.98       100

    accuracy                           0.99       725
   macro avg       0.98      1.00      0.99       725
weighted avg       0.99      0.99      0.99       725

F1 Score: 0.9932
Accuracy: 0.9931


In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = meta_clf.predict_proba(X_meta_test) if hasattr(meta_clf, "predict_proba") else None
_timing = time_inference(predict_end_to_end, X_test_raw[:2000])

row = evaluate(
    y_meta_test,
    stacked_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="SMOTE",
    labels=[0, 1],
    class_names=["benign", "malicious"],
    y_proba=_proba,
    y_train=y_meta_train_bal,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=meta_clf.get_params(),
    seed=RANDOM_STATE,
    n_features=X_test_raw.shape[1],
    test_set="real_heldout_20pct",
    n_train=len(y_meta_train_bal),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")